# 04 — Drug Recommendation

Recommend a set of medications for a patient's current visit using prior visit history.

**Task**: Multilabel classification (predict which drugs from the formulary)  
**Models**: SafeDrug (DDI-constrained), GAMENet, MoleRec  
**Metrics**: Jaccard similarity, PR-AUC, F1, DDI rate (safety metric — lower is better)

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset

ds = SyntheticEHRDataset()
ds.load()

In [ ]:
from pyhealth.tasks import drug_recommendation_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader

task_dataset = ds.dataset.set_task(drug_recommendation_mimic3_fn)
task_dataset.stat()

train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
# SafeDrug uses a DDI constraint to reduce drug-drug interaction rate
# task_dataset.ddi_adj and ddi_mask_H are populated by the drug rec task function
from pyhealth.models import SafeDrug

model = SafeDrug(
    dataset=task_dataset,
    feature_keys=["conditions", "drugs"],
    label_key="drugs",
    mode="multilabel",
    # DDI constraint inputs — produced by the drug recommendation task
    ddi_adj=task_dataset.ddi_adj,
    ddi_mask_H=task_dataset.ddi_mask_H,
)

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model, metrics=["jaccard", "prauc", "f1"])
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    monitor="jaccard",
)

In [ ]:
result = trainer.evaluate(test_loader)

# DDI rate — how often does the model recommend dangerous drug pairs?
# SafeDrug explicitly minimizes this; compare with GAMENet which does not
from pyhealth.metrics import ddi_rate_score
ddi = ddi_rate_score(result["y_prob"], task_dataset.ddi_adj)

import pandas as pd
series = pd.Series(result)
series["ddi_rate"] = ddi
series.drop(["y_prob", "y_true"], errors="ignore").round(4)

In [ ]:
# DrugSafetyChecker wraps DDI checking for a specific patient's medication list
from pyhealth_enterprise.pipelines.drug_safety_checker import DrugSafetyChecker

checker = DrugSafetyChecker(ddi_adj=task_dataset.ddi_adj)
patient_meds = ["A02BC01", "B01AC06", "C07AB03", "C10AA01"]  # example ATC codes

interactions = checker.check_interactions(patient_meds)
high_risk = checker.flag_high_risk_combinations(patient_meds, ddi_threshold=0.3)

print("All pairwise interactions:")
print(interactions)
print(f"\nHigh-risk pairs: {high_risk}")